# Lectura — запуск проекта

Этот notebook поднимает локальное окружение Lectura:
1. Проверяет Node.js, npm и Docker
2. Создаёт `.env` (если нет)
3. Устанавливает зависимости
4. Запускает Postgres / Redis / MinIO через Docker
5. Применяет миграции и seed
6. Стартует Next.js dev server

После успешного запуска открой: **http://127.0.0.1:3000**

> Запускай ячейки **по порядку сверху вниз**.

In [ ]:
import os
import shutil
import socket
import subprocess
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "package.json").exists():
    raise RuntimeError(
        f"package.json не найден в {PROJECT_ROOT}. "
        "Открой notebook из корня репозитория lectura."
    )

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
def run(cmd: str, *, check: bool = True) -> subprocess.CompletedProcess:
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return result


def require_bin(name: str) -> str:
    path = shutil.which(name)
    if not path:
        raise RuntimeError(f"'{name}' не найден в PATH. Установи его и перезапусти kernel.")
    version = run(f"{name} --version", check=False).stdout.splitlines()[0]
    print(f"✓ {name}: {version}")
    return path


require_bin("node")
require_bin("npm")
require_bin("npx")

if shutil.which("docker"):
    require_bin("docker")
    run("docker compose version", check=False)
else:
    print("⚠ Docker не найден — Postgres нужно запустить вручную (brew services start postgresql@14).")

In [ ]:
env_path = PROJECT_ROOT / ".env"
example_path = PROJECT_ROOT / ".env.example"

if not env_path.exists():
    if not example_path.exists():
        raise RuntimeError(".env.example не найден")
    shutil.copy(example_path, env_path)
    print("Создан .env из .env.example — при необходимости добавь API keys.")
else:
    print(".env уже существует — пропускаем.")

In [ ]:
if not (PROJECT_ROOT / "node_modules").exists():
    run("npm install")
else:
    print("node_modules уже есть — пропускаем npm install.")

In [ ]:
if shutil.which("docker"):
    run("docker compose up -d")
else:
    print("Docker пропущен.")

In [ ]:
def port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0


for port in (5433, 5432):
    for attempt in range(30):
        if port_open("127.0.0.1", port):
            print(f"✓ Postgres доступен на порту {port}")
            break
        time.sleep(1)
    else:
        continue
    break
else:
    raise RuntimeError("Postgres не отвечает на 5433/5432. Проверь Docker или DATABASE_URL в .env.")

In [ ]:
run("npx prisma migrate deploy")
run("npm run db:seed")

In [ ]:
PID_FILE = PROJECT_ROOT / ".lectura-dev.pid"

if PID_FILE.exists():
    old_pid = int(PID_FILE.read_text().strip())
    run(f"kill {old_pid}", check=False)
    PID_FILE.unlink(missing_ok=True)

for port in (3000, 3001, 3002, 3003):
    run(f"lsof -ti :{port} | xargs kill -9", check=False)

log_file = open(PROJECT_ROOT / ".lectura-dev.log", "w")
proc = subprocess.Popen(
    "WATCHPACK_POLLING=true npm run dev",
    shell=True,
    cwd=PROJECT_ROOT,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
PID_FILE.write_text(str(proc.pid))

print(f"Dev server запущен (pid={proc.pid}). Лог: .lectura-dev.log")
print("Ждём http://127.0.0.1:3000 ...")

for _ in range(60):
    if port_open("127.0.0.1", 3000):
        print("\n✓ Lectura готова: http://127.0.0.1:3000")
        break
    time.sleep(1)
else:
    print("\n⚠ Сервер ещё стартует. Проверь .lectura-dev.log если страница не открывается.")

## Остановить dev server

Выполни ячейку ниже, когда закончишь работу.

In [ ]:
PID_FILE = PROJECT_ROOT / ".lectura-dev.pid"

if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    run(f"kill {pid}", check=False)
    PID_FILE.unlink(missing_ok=True)
    print(f"Остановлен процесс {pid}")
else:
    print("PID файл не найден — возможно сервер уже остановлен.")

for port in (3000, 3001, 3002, 3003):
    run(f"lsof -ti :{port} | xargs kill -9", check=False)